In [ ]:
import pandas as pd
import numpy as np
import json
import itertools
import warnings
warnings.filterwarnings("ignore")

from prophet import Prophet
from statsmodels.tsa.statespace.sarimax import SARIMAX

INPUT_DIR_NB1 = "<link-to-the-1st-notebook>"
INPUT_DIR_NB2 = "<link-to-the-2nd-notebook>"

subset = pd.read_parquet(f"{INPUT_DIR_NB2}/retailcast_features.parquet")
with open(f"{INPUT_DIR_NB1}/subset_config.json") as f:
    config = json.load(f)

subset["date"] = pd.to_datetime(subset["date"])
selected_stores = config["selected_stores"]
selected_families = config["selected_families"]
representative_combos = [tuple(x) for x in config["representative_combos"]]

In [ ]:
HORIZON = 15
N_FOLDS = 4

def get_walkforward_folds(all_dates, horizon=HORIZON, n_folds=N_FOLDS):
    """
    Returns list of (train_end_date, test_start_date, test_end_date) for n_folds
    expanding-window folds, plus the final holdout window (most recent `horizon` days).
    Folds are ordered oldest -> newest, ending right before the holdout.
    """
    all_dates = sorted(all_dates)
    last_date = all_dates[-1]
    holdout_start = last_date - pd.Timedelta(days=horizon - 1)

    folds = []
    for i in range(n_folds, 0, -1):
        test_end = holdout_start - pd.Timedelta(days=(i - 1) * horizon + 1)
        test_start = test_end - pd.Timedelta(days=horizon - 1)
        train_end = test_start - pd.Timedelta(days=1)
        folds.append((train_end, test_start, test_end))

    holdout = (holdout_start - pd.Timedelta(days=1), holdout_start, last_date)
    return folds, holdout

all_dates = subset["date"].unique()
folds, holdout = get_walkforward_folds(all_dates)
print("CV folds (train_end, test_start, test_end):")
for f in folds:
    print(f)
print("Holdout:", holdout)

In [ ]:
def mape(actual, forecast):
    actual, forecast = np.array(actual), np.array(forecast)
    mask = actual != 0
    return np.mean(np.abs((actual[mask] - forecast[mask]) / actual[mask])) * 100

def wape(actual, forecast):
    actual, forecast = np.array(actual), np.array(forecast)
    return np.sum(np.abs(actual - forecast)) / np.sum(np.abs(actual)) * 100

def mase(actual, forecast, train_series, seasonal_period=7):
    actual, forecast = np.array(actual), np.array(forecast)
    train_series = np.array(train_series)
    naive_errors = np.abs(train_series[seasonal_period:] - train_series[:-seasonal_period])
    scale = np.mean(naive_errors) if len(naive_errors) > 0 else 1.0
    return np.mean(np.abs(actual - forecast)) / scale if scale != 0 else np.nan

In [ ]:
prophet_results = []

for store_nbr in selected_stores:
    for family in selected_families:
        series_df = subset[
            (subset["store_nbr"] == store_nbr) & (subset["family"] == family)
        ][["date", "sales", "onpromotion", "is_holiday"]].sort_values("date")
        series_df = series_df.rename(columns={"date": "ds", "sales": "y"})

        for fold_idx, (train_end, test_start, test_end) in enumerate(folds + [holdout]):
            fold_label = f"fold_{fold_idx+1}" if fold_idx < len(folds) else "holdout"

            train_df = series_df[series_df["ds"] <= train_end]
            test_df = series_df[(series_df["ds"] >= test_start) & (series_df["ds"] <= test_end)]
            if len(train_df) < 60 or len(test_df) == 0:
                continue

            m = Prophet(weekly_seasonality=True, yearly_seasonality=True, daily_seasonality=False)
            m.add_regressor("onpromotion")
            m.add_regressor("is_holiday")
            m.fit(train_df[["ds", "y", "onpromotion", "is_holiday"]])

            future = test_df[["ds", "onpromotion", "is_holiday"]].copy()
            forecast = m.predict(future)

            actual = test_df["y"].values
            pred = forecast["yhat"].clip(lower=0).values

            prophet_results.append({
                "store_nbr": store_nbr, "family": family, "fold": fold_label,
                "mape": mape(actual, pred), "wape": wape(actual, pred),
                "mase": mase(actual, pred, train_df["y"].values),
            })

prophet_results_df = pd.DataFrame(prophet_results)
print(prophet_results_df.groupby("fold")[["mape", "wape", "mase"]].mean())

In [ ]:
def grid_search_sarima(train_series, seasonal_period=7):
    """Small grid search over (p,d,q)(P,D,Q,s), picks lowest AIC. Kept small for CPU feasibility."""
    p = d = q = range(0, 2)
    P = D = Q = range(0, 2)
    best_aic = np.inf
    best_order = None
    best_seasonal_order = None

    for order in itertools.product(p, d, q):
        for seasonal_order in itertools.product(P, D, Q):
            try:
                model = SARIMAX(
                    train_series, order=order,
                    seasonal_order=(*seasonal_order, seasonal_period),
                    enforce_stationarity=False, enforce_invertibility=False,
                )
                fit = model.fit(disp=False)
                if fit.aic < best_aic:
                    best_aic = fit.aic
                    best_order = order
                    best_seasonal_order = (*seasonal_order, seasonal_period)
            except Exception:
                continue
    return best_order, best_seasonal_order, best_aic

sarima_results = []

for store_nbr, family in representative_combos:
    series_df = subset[
        (subset["store_nbr"] == store_nbr) & (subset["family"] == family)
    ].sort_values("date").set_index("date")["sales"].asfreq("D").fillna(0)

    # Order selection done ONCE per series on data up to the first fold's train_end
    initial_train = series_df[series_df.index <= folds[0][0]]
    best_order, best_seasonal_order, best_aic = grid_search_sarima(initial_train)
    print(f"Store {store_nbr}, {family}: best order={best_order}, seasonal={best_seasonal_order}, AIC={best_aic:.1f}")

    for fold_idx, (train_end, test_start, test_end) in enumerate(folds + [holdout]):
        fold_label = f"fold_{fold_idx+1}" if fold_idx < len(folds) else "holdout"
        train_series = series_df[series_df.index <= train_end]
        test_series = series_df[(series_df.index >= test_start) & (series_df.index <= test_end)]
        if len(train_series) < 60 or len(test_series) == 0:
            continue

        try:
            model = SARIMAX(
                train_series, order=best_order, seasonal_order=best_seasonal_order,
                enforce_stationarity=False, enforce_invertibility=False,
            )
            fit = model.fit(disp=False)
            pred = fit.forecast(steps=len(test_series))
            pred = np.clip(pred.values, 0, None)
            actual = test_series.values

            sarima_results.append({
                "store_nbr": store_nbr, "family": family, "fold": fold_label,
                "order": str(best_order), "seasonal_order": str(best_seasonal_order),
                "mape": mape(actual, pred), "wape": wape(actual, pred),
                "mase": mase(actual, pred, train_series.values),
            })
        except Exception as e:
            print(f"SARIMA fit failed for {store_nbr}/{family}/{fold_label}: {e}")

sarima_results_df = pd.DataFrame(sarima_results)
print(sarima_results_df)

In [ ]:
sarima_results_df.to_csv("/kaggle/working/sarima_results.csv", index=False)
prophet_results_df.to_csv("/kaggle/working/prophet_results.csv", index=False)
print("Saved: prophet_results.csv, sarima_results.csv")